# 🇮🇳 Chatterbox Maithili Fine-tuning

This notebook fine-tunes the Chatterbox TTS model on Maithili speech data using Kaggle T4 x2 GPUs.

**Important**: After running Cell 1 (setup), **restart the Kaggle session** before running Cell 2+.

In [1]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

# 1. Setup Environment
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
os.environ["TRAINING_ENV"] = "kaggle"

# 2. Clone Repository
REPO_URL = "https://github.com/Firojpaudel/chatterbox-nepali.git"
REPO_DIR = "/kaggle/working/chatterbox-nepali"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Updating repository...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)

# 3. Install Dependencies
# =========================================================================
# KEY FIXES:
#   1. Install chatterbox FIRST (pins numpy<2.0 → 1.26.x)
#   2. Then install pandas<2.3 (compatible with numpy 1.x binaries)
#   3. Remove torchcodec (Kaggle's version is broken, causes FFmpeg crash
#      when datasets lib tries to decode audio — we use soundfile instead)
# =========================================================================
print("Installing dependencies...")

# Step 0: Kill torchcodec FIRST — it's pre-installed on Kaggle but broken
# (compiled against wrong FFmpeg/PyTorch). datasets will fall back to soundfile.
subprocess.run(["pip", "uninstall", "-y", "torchcodec"], check=False)

# Step 1: PyTorch (already present on Kaggle, but pin versions)
subprocess.run([
    "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
    "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)

# Step 2: Install chatterbox (pins numpy<2.0.0 → gets numpy 1.26.4)
subprocess.run(["pip", "install", "-e", "."], check=True)

# Step 3: Force-install pandas compatible with numpy 1.x
# pandas 3.x needs numpy 2.x binaries → crash. pandas <2.3 works with numpy 1.26.x
subprocess.run([
    "pip", "install",
    "pandas>=2.0,<2.3",
    "--force-reinstall", "--no-deps"
], check=True)

# Step 4: datasets + its deps (will use soundfile for audio since torchcodec is gone)
subprocess.run(["pip", "install", "datasets", "soundfile"], check=True)

# Step 5: Other training dependencies
subprocess.run([
    "pip", "install",
    "omegaconf", "conformer", "s3tokenizer", "safetensors",
    "pyloudnorm", "wandb", "huggingface-hub", "librosa", "soundfile"
], check=True)
subprocess.run(["pip", "install", "git+https://github.com/resemble-ai/Perth.git@master"], check=True)

print("\n--- Chatterbox Setup Complete ---")

Cloning repository from https://github.com/Firojpaudel/chatterbox-nepali.git...


Cloning into '/kaggle/working/chatterbox-nepali'...


Installing dependencies...
Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Running command git clone --filter=blob:none --quiet https://github.com/resemble-ai/Perth.git /tmp/pip-install-8a5jtju1/resemble-perth_ca74f13c2fc04f57be4f03c8b9cd4d8b


  Resolved https://github.com/resemble-ai/Perth.git to commit ce86c49d029f42272c1902eccb675556b9ed2330
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 32.2 MB/s eta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 101.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3
  Cloning https://github.com/resemble-ai/Perth.git (to revision master) to /tmp/pip-req-build-yyxxmeut


  Running command git clone --filter=blob:none --quiet https://github.com/resemble-ai/Perth.git /tmp/pip-req-build-yyxxmeut


  Resolved https://github.com/resemble-ai/Perth.git to commit ce86c49d029f42272c1902eccb675556b9ed2330
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'

--- Chatterbox Setup Complete ---


In [2]:
%%writefile /kaggle/working/chatterbox-nepali/download_ckpt.py
import os
from huggingface_hub import hf_hub_download

print("Downloading base model to warm-start Maithili...")
ckpt_path = hf_hub_download(
    repo_id="Firoj112/chatterbox-nepali-runs",
    filename="t3_nepali_epoch_30.pt",
    token=os.environ.get("HF_TOKEN")
)
dest = "t3_nepali_epoch_30.pt"
if os.path.exists(dest):
    os.remove(dest)
os.symlink(ckpt_path, dest)
print(f"✅ Checkpoint ready at {dest}")


Writing /kaggle/working/chatterbox-nepali/prepare_data.py


In [3]:
!python /kaggle/working/chatterbox-nepali/download_ckpt.py
!python /kaggle/working/chatterbox-nepali/prepare_kaggle_maithili.py --output_dir finetuning_data_maithili

✅ numpy  1.26.4  (expect 1.26.x)
✅ pandas 2.2.3  (expect 2.x, < 2.3)
✅ datasets loaded, torchcodec disabled
✅ datasets imported OK
CWD: /kaggle/working/chatterbox-nepali
t3_nepali_epoch_30.pt:  38%|██████          | 805M/2.14G [00:12<00:20, 63.9MB/s]
✅ Checkpoint ready at t3_nepali_epoch_30.pt
Fetching 8 files: 100%|███████████████████████████| 8/8 [00:11<00:00,  1.40s/it]
Download complete: : 1.46GB [00:11, 113MB/s]              Manifests: ['validation_manifest.jsonl', 'train_manifest.jsonl', 'test_manifest.jsonl']
Found 5 parquet files
Processing test-00000-of-00001.parquet (401 rows)...

test-00000-of-00001.parquet: 100%|██████████| 401/401 [00:00<00:00, 3179.74it/s]
Processing train-00000-of-00003.parquet (2401 rows)...

train-00000-of-00003.parquet: 100%|███████| 2401/2401 [00:00<00:00, 3595.21it/s]
Processing train-00001-of-00003.parquet (2400 rows)...

train-00001-of-00003.parquet: 100%|███████| 2400/2400 [00:00<00:00, 3685.28it/s]
Processing train-00002-of-00003.parquet (2400 r

In [4]:
# ============================================================
# Cell 5: 🚀 START DISTRIBUTED TRAINING (T4 x 2)
# ============================================================
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  torchrun --nproc_per_node=2 src/chatterbox/train_maithili.py \
    --manifest finetuning_data_maithili/maithili_manifest.jsonl \
    --wav_dir finetuning_data_maithili/wavs \
    --resume_t3_weights t3_nepali_epoch_30.pt \
    --batch_size 4 \
    --accum_steps 4 \
    --epochs 50 \
    --lr 2e-5 \
    --save_every 5 \
    --num_workers 2 \
    --distributed \
    --fp16 \
    --push_to_hub Firoj112/chatterbox-maithili-runs


W0501 05:31:14.407000 753 torch/distributed/run.py:792] 
W0501 05:31:14.407000 753 torch/distributed/run.py:792] *****************************************
W0501 05:31:14.407000 753 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0501 05:31:14.407000 753 torch/distributed/run.py:792] *****************************************
Fetching 6 files:   0%|                                   | 0/6 [00:00<?, ?it/s]wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.

wandb: ⣻ setting up run y5mzqu6e (0.2s)
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/chatterbox-nepali/wandb/run-20260501_053136-y5mzqu6e
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run floral-smoke-9
wandb: ⭐️ View project at https://wandb.

In [5]:
%%writefile /kaggle/working/chatterbox-nepali/push_model.py
# ============================================================
# 📤 Push final model to Hugging Face
# ============================================================
from huggingface_hub import HfApi
import os
import glob

api = HfApi()
repo_id = "Firoj112/chatterbox-maithili-runs"
token = os.environ.get("HF_TOKEN")

final_model = "t3_mtl_maithili_final.safetensors"

if not token:
    print("❌ HF_TOKEN not set.")
elif not os.path.exists(final_model):
    print(f"❌ {final_model} not found.")
    checkpoints = sorted(glob.glob("t3_maithili_epoch_*.pt"))
    if checkpoints:
        latest = checkpoints[-1]
        print(f"📤 Pushing latest checkpoint instead: {latest}")
        api.upload_file(path_or_fileobj=latest, path_in_repo=latest, repo_id=repo_id, repo_type="model", token=token)
        print("✅ Pushed successfully!")
else:
    print(f"📤 Pushing {final_model} to {repo_id}...")
    api.upload_file(path_or_fileobj=final_model, path_in_repo=final_model, repo_id=repo_id, repo_type="model", token=token)
    print("✅ Pushed successfully!")


Writing /kaggle/working/chatterbox-nepali/push_model.py


In [6]:
from huggingface_hub import HfApi
import os
import glob

api = HfApi()
repo_id = "Firoj112/chatterbox-maithili-runs"
token = os.environ.get("HF_TOKEN")

final_model = "t3_mtl_maithili_final.safetensors"

if not token:
    print("❌ HF_TOKEN not set.")
elif not os.path.exists(final_model):
    checkpoints = sorted(glob.glob("t3_maithili_epoch_*.pt"))
    if checkpoints:
        latest = checkpoints[-1]
        print(f"📤 Pushing {latest}...")
        api.upload_file(path_or_fileobj=latest, path_in_repo=latest, repo_id=repo_id, repo_type="model", token=token)
else:
    api.upload_file(path_or_fileobj=final_model, path_in_repo=final_model, repo_id=repo_id, repo_type="model", token=token)


📤 Pushing t3_mtl_nepali_final.safetensors to Firoj112/chatterbox-nepali-runs...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Successfully pushed to Hugging Face!
